In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# importations
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,mean_squared_error,r2_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVC
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBClassifier
from xgboost import XGBRegressor
from catboost import CatBoostClassifier
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(f"{path}/Q1_data.csv")
df.head()

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
df["Delivery_Time"].hist()

In [ ]:
#
print(f"Sahape is: {df.shape}")

In [ ]:
# Task 1: Write your code here:
df_new = df.copy().drop(columns=["Order_ID"])
df_new.head()

In [ ]:
print(f"Before: {df_new.isna().sum()}")

In [ ]:
# Task 2: Write your code here:
df_new.isna().sum()
df_new = df_new.dropna(subset=["Delivery_Time", "Weather", "Traffic_Level", "Time_of_Day", "Courier_Experience_yrs"])
print(f"After: {df_new.isna().sum()}")

In [ ]:
print(f"Sahpe after dropping: {df_new.shape}")

In [ ]:
# Task 3: Write your code here:
print(df_new.duplicated().sum())
df_new = df_new.drop_duplicates()
print(f"Sahpe after dropping: {df_new.shape}")
df.head()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

encoder=LabelEncoder()

for col in df_new.select_dtypes(include=["object"]).columns:
    df_new[col]=encoder.fit_transform(df_new[col])

df_new.head()

In [ ]:
# Task 5: Write your code here:
features = df_new.drop(columns=["Delivery_Time"]).columns
features

sclaer = StandardScaler()

df_new[features] = sclaer.fit_transform(df_new[features])
df_new.head()

In [ ]:
# Task 6: Write your code here:
# this is a regression problem (:

In [ ]:
# Task 1: Write your code here:

X = df_new[features]
y = df_new["Delivery_Time"]
X.shape

In [ ]:
model = RandomForestRegressor()
model

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error,mean_squared_error

kf=KFold(n_splits=5,shuffle=True,random_state=42)

scores_mae=[]

# scores_rmse=[]
for train_index,test_index in kf.split(X):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train,y_train)

    y_pred=model.predict(X_test)

    scores_mae.append(mean_absolute_error(y_test, y_pred))
    # scores_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))

print(f"{model} MAE Score: {np.mean(scores_mae)}")
# print(f"{model} RMSE Score: {np.mean(scores_rmse)}")

In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(10, 6))
plt.plot(scores_mae, label='Training', color='purple')
plt.title('Loss Curve')
plt.xlabel('Iteration')
plt.ylabel('Categorical Cross-Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task 2: Write your code here:

coef = model.feature_importances_

absolute_coef = np.abs(coef)
sorted_idx = np.argsort(absolute_coef)

features = X.columns

plt.barh(features[sorted_idx], coef[sorted_idx])
plt.title(f"Random forest Coefficients")
plt.xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()


In [ ]:
!pip install catboost

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score, root_mean_squared_error

In [ ]:
# Task Bonus: Write your code here:


models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

n_splits = 5

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)

In [ ]:
from sklearn.metrics import root_mean_squared_error
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MSE:  {np.mean(all_results[model_name]['mse']):.4f}")
  print(f"  RMSE: {np.mean(all_results[model_name]['rmse']):.4f}")
  print(f"  R2:    {np.mean(all_results[model_name]['r2']):.4f}")